# preview_density_labels.py

**Converted from:** `/Users/songkarn/locarb/biomass/handheld-lidar-slam-toolbox/yolov11/scripts/preview_density_labels.py`

**Description:** This notebook was automatically generated from the Python script for easier execution and modification on another machine.

## Imports

In [ ]:
#!/usr/bin/env python3

## Main Code

In [ ]:
"""Preview density labels with bounding boxes"""import numpy as npimport cv2from pathlib import Pathimport random# ConfigurationDATASET_DIR = Path("yolov11/dataset")SPLITS = ['train', 'test', 'val']NUM_SAMPLES = 9OUTPUT_DIR = Path("yolov11/dataset/previews")BOX_COLOR = (0, 255, 0)  # GreenBOX_THICKNESS = 2def load_yolo_labels(label_path):    """Load YOLO format labels: class x y w h (all normalized)"""    if not label_path.exists():        return []        labels = []    with open(label_path, 'r') as f:        for line in f:            parts = line.strip().split()            if len(parts) == 5:                cls = int(parts[0])                x, y, w, h = map(float, parts[1:])                labels.append((cls, x, y, w, h))    return labelsdef yolo_to_bbox(x_center, y_center, width, height, img_width, img_height):    """Convert YOLO format to pixel bbox"""    x_center_px = x_center * img_width    y_center_px = y_center * img_height    w_px = width * img_width    h_px = height * img_height        x1 = int(x_center_px - w_px / 2)    y1 = int(y_center_px - h_px / 2)    x2 = int(x_center_px + w_px / 2)    y2 = int(y_center_px + h_px / 2)        return x1, y1, x2, y2def draw_labels_on_image(img, labels):    """Draw bounding boxes on image"""    img_h, img_w = img.shape[:2]    img_copy = img.copy()        for cls, x, y, w, h in labels:        x1, y1, x2, y2 = yolo_to_bbox(x, y, w, h, img_w, img_h)                # Draw rectangle        cv2.rectangle(img_copy, (x1, y1), (x2, y2), BOX_COLOR, BOX_THICKNESS)                # Draw class label        label_text = 'tree'        (text_w, text_h), _ = cv2.getTextSize(            label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1        )                cv2.rectangle(            img_copy,            (x1, y1 - text_h - 4),            (x1 + text_w + 4, y1),            BOX_COLOR,            -1        )                cv2.putText(            img_copy,            label_text,            (x1 + 2, y1 - 2),            cv2.FONT_HERSHEY_SIMPLEX,            0.5,            (255, 255, 255),            1,            cv2.LINE_AA        )        return img_copydef create_preview_grid(images, rows=3, cols=3):    """Create a grid of preview images"""    if not images:        return None        # Find the most common size or use first image size    sizes = [img.shape[:2] for img in images]    img_h, img_w = sizes[0]        # Resize all images to the same size (use first image size as reference)    resized_images = []    for img in images:        if img.shape[:2] != (img_h, img_w):            img = cv2.resize(img, (img_w, img_h))        resized_images.append(img)        # Create empty grid    grid = np.zeros((rows * img_h, cols * img_w, 3), dtype=np.uint8)        # Fill grid    for idx, img in enumerate(resized_images):        if idx >= rows * cols:            break        row = idx // cols        col = idx % cols        y1 = row * img_h        y2 = y1 + img_h        x1 = col * img_w        x2 = x1 + img_w        grid[y1:y2, x1:x2] = img        return griddef preview_split(split):    """Preview samples from a split"""    img_dir = DATASET_DIR / 'images' / split / 'density'    label_dir = DATASET_DIR / 'labels' / split / 'density'        print(f"\n=== Processing {split.upper()} split ===")    print(f"Image dir: {img_dir}")    print(f"Label dir: {label_dir}")        if not img_dir.exists():        print(f"  ⚠ Image directory not found, skipping")        return        if not label_dir.exists():        print(f"  ⚠ Label directory not found, skipping")        return        # Get all image files    img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))        if not img_files:        print(f"  ⚠ No images found, skipping")        return        # Sample images    if len(img_files) < NUM_SAMPLES:        print(f"  Found {len(img_files)} images (showing all)")        sampled_files = img_files    else:        sampled_files = random.sample(img_files, NUM_SAMPLES)        print(f"  Found {len(img_files)} images (showing {NUM_SAMPLES} random samples)")        # Create previews    preview_images = []    stats = {'total_boxes': 0, 'images_with_labels': 0, 'images_processed': 0}        for img_path in sampled_files:        # Load image        img = cv2.imread(str(img_path))        if img is None:            print(f"  ⚠ Could not load {img_path.name}")            continue                # Load labels        label_path = label_dir / f"{img_path.stem}.txt"        labels = load_yolo_labels(label_path)                # Stats        stats['images_processed'] += 1        if labels:            stats['images_with_labels'] += 1            stats['total_boxes'] += len(labels)                # Draw labels        img_with_labels = draw_labels_on_image(img, labels)                # Add image name as title        title = img_path.stem[:30]  # Truncate long names        if len(labels) > 0:            title += f" ({len(labels)} trees)"        else:            title += " (no labels)"                cv2.putText(            img_with_labels,            title,            (10, 25),            cv2.FONT_HERSHEY_SIMPLEX,            0.6,            (255, 255, 0),            2,            cv2.LINE_AA        )                preview_images.append(img_with_labels)        if not preview_images:        print(f"  ⚠ No valid images to preview")        return        # Create grid    grid = create_preview_grid(preview_images)        # Save    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)    output_path = OUTPUT_DIR / f'{split}_density_preview.jpg'    cv2.imwrite(str(output_path), grid)    print(f"  ✓ Saved preview to: {output_path}")        # Print stats    print(f"  Statistics:")    print(f"    - Images processed: {stats['images_processed']}")    print(f"    - Images with labels: {stats['images_with_labels']}")    print(f"    - Total bounding boxes: {stats['total_boxes']}")    if stats['images_with_labels'] > 0:        avg_boxes = stats['total_boxes'] / stats['images_with_labels']        print(f"    - Avg boxes per labeled image: {avg_boxes:.1f}")def main():    print("=" * 60)    print("YOLO Density Dataset Label Preview Generator")    print("=" * 60)        for split in SPLITS:        preview_split(split)        print(f"\n{'=' * 60}")    print(f"✓ Done! Check previews in: {OUTPUT_DIR}")    print(f"{'=' * 60}\n")if __name__ == '__main__':    main()